# Does the embedding ranking survive hybrid retrieval?

Phase 15 re-scored seven embedding models on `eval_v3` and found Gemini beating the incumbent Cohere `embed-v4.0` by 12 questions to 3 (sign test, p = 0.035). Every one of those numbers is **semantic-only**.

Production is not semantic-only. `src/retrieval.py` runs **hybrid RRF** — a Pinecone semantic ranking fused with a BM25 keyword ranking at `RRF_K=5`, `RRF_W=0.85`, `BM25_K1=1.5`, `BM25_B=0.75`, each ranker contributing 50 candidates.

That matters because BM25 covers exactly the case a natural-language question makes harder: the user typing a distinctive term that appears verbatim in the article. An embedding model's *marginal* value is whatever BM25 does not already find — so a model that wins alone can win by less, or not at all, once fused.

This notebook answers that before anything is switched. It is the last cheap check standing between the Phase 15 result and a decision that costs a new Pinecone index and a code change.

## Zero cost

No API calls at all. Corpus **and** query vectors for all seven models are already in `eval/emb_cache/` from Phases 13 and 15, and BM25 runs locally off `chunks_articles.parquet`. The whole notebook is arithmetic on cached arrays.

## Two things it deliberately checks

1. **The ranking under production's exact constants.** Does Gemini still beat Cohere when BM25 is in the loop?
2. **Whether that conclusion depends on `RRF_W`.** `w = 0.85` was fitted in Phase 6 *on clause chunks, with Cohere as the semantic ranker*. Using a Cohere-tuned weight to judge Gemini is a real confound, so `w` is swept from 0 (pure keyword) to 1 (pure semantic). If the winner flips somewhere in that range, the answer is "it depends on tuning"; if it does not, the result is robust to a parameter that was never fitted for it.

In [1]:
import sys
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CACHE_DIR = PROJECT / "eval" / "emb_cache"
sys.path.insert(0, str(PROJECT))

# Reuse production's own Arabic normalisation and tokenizer rather than re-implementing
# them -- a subtly different tokenizer here would make this test measure the wrong thing.
from src.retrieval import (tokenize, RRF_K, RRF_W, BM25_K1, BM25_B, CANDIDATE_TOP_K)

chunks = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks["article_number"] = chunks["article_number"].astype(str)
eval_q = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v3.jsonl", encoding="utf-8")])

CORPUS_HASH = hashlib.md5(
    "".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode("utf-8")).hexdigest()[:12]
QUERY_HASH = hashlib.md5(
    "".join(eval_q["question_id"] + "\x00" + eval_q["question_text"]).encode("utf-8")).hexdigest()[:12]

print(f"production constants -- RRF_K={RRF_K} RRF_W={RRF_W} "
      f"BM25_K1={BM25_K1} BM25_B={BM25_B} candidates={CANDIDATE_TOP_K}")
print(f"corpus {CORPUS_HASH} ({len(chunks)} articles) | queries {QUERY_HASH} ({len(eval_q)})")

bm25 = BM25Okapi([tokenize(t) for t in chunks["text"]], k1=BM25_K1, b=BM25_B)
print("BM25 index built from the same parquet production reads")

production constants -- RRF_K=5 RRF_W=0.85 BM25_K1=1.5 BM25_B=0.75 candidates=50
corpus 88dbec8c163f (531 articles) | queries 6e8e3718215d (60)
BM25 index built from the same parquet production reads


## Scoring

Identical fusion to `Retriever.retrieve()`: take the top `CANDIDATE_TOP_K` from each ranker, score each document `w/(k+rank_semantic) + (1-w)/(k+rank_keyword)`, sort, cut to 10.

In [2]:
TOP_K = 10
N = len(chunks)

# BM25 ranks depend only on the question, not on the embedding model -- compute once.
bm25_ranks = []
for q in eval_q["question_text"]:
    scores = bm25.get_scores(tokenize(q))
    bm25_ranks.append(np.argsort(-scores)[:CANDIDATE_TOP_K])


def rrf_ranked(sem_idx, kw_idx, w, k=RRF_K, top_k=TOP_K):
    s = {}
    for rank, i in enumerate(sem_idx, start=1):
        s[i] = s.get(i, 0.0) + w / (k + rank)
    for rank, i in enumerate(kw_idx, start=1):
        s[i] = s.get(i, 0.0) + (1 - w) / (k + rank)
    return [i for i, _ in sorted(s.items(), key=lambda kv: kv[1], reverse=True)[:top_k]]


def score_run(order_per_q):
    rows = []
    for qi, q in enumerate(eval_q.itertuples(index=False)):
        correct = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, idx in enumerate(order_per_q[qi], start=1):
            c = chunks.iloc[idx]
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in correct):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({"question_id": q.question_id, "rank": rank, "full_coverage": found == correct})
    return pd.DataFrame(rows)


import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import metrics as _metrics

def metrics(df):
    return _metrics(df, include_n=False)


def semantic_ranks(model):
    corpus = torch.from_numpy(np.load(CACHE_DIR / f"{model}__{CORPUS_HASH}__corpus.npy"))
    queries = torch.from_numpy(np.load(CACHE_DIR / f"{model}__{QUERY_HASH}__query.npy"))
    sims = queries @ corpus.T
    return torch.topk(sims, k=CANDIDATE_TOP_K, dim=1).indices.numpy()


MODELS = ["gemini-embedding-001", "openai-text-embedding-3-large", "cohere-embed-v4",
          "bge-m3", "multilingual-e5-large-instruct", "openai-text-embedding-3-small",
          "qwen3-embedding-0.6b"]
MODELS = [m for m in MODELS if (CACHE_DIR / f"{m}__{QUERY_HASH}__query.npy").exists()]
SEM = {m: semantic_ranks(m) for m in MODELS}
print("loaded cached vectors for", len(MODELS), "models -- no API calls made")

loaded cached vectors for 7 models -- no API calls made


## Semantic-only vs hybrid, at production's `w = 0.85`

Plus a **BM25-only** row, which is the same for every model and shows what the keyword half contributes on its own.

In [3]:
bm25_only = score_run([list(r[:TOP_K]) for r in bm25_ranks])
print(f"BM25 alone (no embeddings):  recall@1 {metrics(bm25_only)['recall@1']:.3f}  "
      f"recall@5 {metrics(bm25_only)['recall@5']:.3f}  MRR {metrics(bm25_only)['mrr']:.3f}\n")

rows, hybrid_details = [], {}
for m in MODELS:
    sem_det = score_run([list(SEM[m][qi][:TOP_K]) for qi in range(len(eval_q))])
    hyb_order = [rrf_ranked(SEM[m][qi], bm25_ranks[qi], RRF_W) for qi in range(len(eval_q))]
    hyb_det = score_run(hyb_order)
    hybrid_details[m] = hyb_det
    s, h = metrics(sem_det), metrics(hyb_det)
    rows.append({"model": m,
                 "sem_recall@1": s["recall@1"], "hyb_recall@1": h["recall@1"],
                 "sem_recall@5": s["recall@5"], "hyb_recall@5": h["recall@5"],
                 "sem_mrr": s["mrr"], "hyb_mrr": h["mrr"],
                 "d_mrr": h["mrr"] - s["mrr"]})

cmp = pd.DataFrame(rows).sort_values("hyb_mrr", ascending=False)
pd.set_option("display.width", 170)
print("=" * 92)
print(f"SEMANTIC-ONLY vs HYBRID RRF (w={RRF_W}) on eval_v3")
print("=" * 92)
print(cmp.round(3).to_string(index=False))
cmp.to_csv(PROJECT / "eval" / "embedding_hybrid_v3_summary.csv", index=False, encoding="utf-8-sig")

BM25 alone (no embeddings):  recall@1 0.317  recall@5 0.600  MRR 0.437



SEMANTIC-ONLY vs HYBRID RRF (w=0.85) on eval_v3
                         model  sem_recall@1  hyb_recall@1  sem_recall@5  hyb_recall@5  sem_mrr  hyb_mrr  d_mrr
          gemini-embedding-001         0.867         0.883         1.000         1.000    0.922    0.931  0.008
 openai-text-embedding-3-large         0.733         0.733         0.950         0.950    0.837    0.838  0.001
               cohere-embed-v4         0.717         0.733         1.000         1.000    0.828    0.838  0.010
                        bge-m3         0.667         0.717         0.967         0.967    0.803    0.835  0.032
multilingual-e5-large-instruct         0.633         0.633         0.900         0.933    0.741    0.751  0.010
 openai-text-embedding-3-small         0.550         0.533         0.883         0.883    0.694    0.692 -0.002
          qwen3-embedding-0.6b         0.517         0.533         0.833         0.833    0.657    0.674  0.017


## Is the Gemini-over-Cohere result still there under hybrid?

Same paired sign test as Phase 15, now on hybrid ranks.

In [4]:
from math import comb


def sign_test(a, b, details):
    ha = (details[a]["rank"] == 1).values
    hb = (details[b]["rank"] == 1).values
    n10, n01 = int((ha & ~hb).sum()), int((~ha & hb).sum())
    n = n10 + n01
    p = 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(0, min(n10, n01) + 1)) / 2 ** n)
    return n10, n01, p


PAIRS = [("gemini-embedding-001", "cohere-embed-v4"),
         ("gemini-embedding-001", "openai-text-embedding-3-large"),
         ("openai-text-embedding-3-large", "cohere-embed-v4")]

print(f"{'comparison':62s} {'A':>4} {'B':>4} {'p':>8}")
for a, b in PAIRS:
    if a in hybrid_details and b in hybrid_details:
        w, l, p = sign_test(a, b, hybrid_details)
        print(f"{a + '  vs  ' + b:62s} {w:>4} {l:>4} {p:>8.4f}")

comparison                                                        A    B        p
gemini-embedding-001  vs  cohere-embed-v4                        11    2   0.0225
gemini-embedding-001  vs  openai-text-embedding-3-large          12    3   0.0352
openai-text-embedding-3-large  vs  cohere-embed-v4                8    8   1.0000


## Does the answer depend on `RRF_W`?

`w = 0.85` was fitted in Phase 6 on clause chunks with Cohere as the semantic ranker, so it is not a neutral setting for judging a different embedding model. Sweeping it from 0 (pure keyword) to 1 (pure semantic) shows whether the conclusion is an artifact of that one value.

In [5]:
WS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
sweep = []
for m in MODELS:
    for w in WS:
        det = score_run([rrf_ranked(SEM[m][qi], bm25_ranks[qi], w) for qi in range(len(eval_q))])
        sweep.append({"model": m, "w": w, **metrics(det)})
sw = pd.DataFrame(sweep)
sw.to_csv(PROJECT / "eval" / "embedding_hybrid_v3_wsweep.csv", index=False, encoding="utf-8-sig")

print("MRR by RRF weight (w=0 pure BM25, w=1 pure semantic)")
print(sw.pivot(index="w", columns="model", values="mrr").round(3).to_string())

print("\nbest w per model, by MRR:")
best = sw.loc[sw.groupby("model")["mrr"].idxmax()][["model", "w", "recall@1", "recall@5", "mrr"]]
print(best.sort_values("mrr", ascending=False).round(3).to_string(index=False))

piv = sw.pivot(index="w", columns="model", values="mrr")
winner = piv.idxmax(axis=1)
print("\nwinner at each w:")
for w, mdl in winner.items():
    print(f"  w={w:<5} {mdl}")
print(f"\ndistinct winners across the sweep: {sorted(set(winner))}")

MRR by RRF weight (w=0 pure BM25, w=1 pure semantic)
model  bge-m3  cohere-embed-v4  gemini-embedding-001  multilingual-e5-large-instruct  openai-text-embedding-3-large  openai-text-embedding-3-small  qwen3-embedding-0.6b
w                                                                                                                                                                       
0.00    0.437            0.437                 0.437                           0.437                          0.437                          0.437                 0.437
0.20    0.502            0.506                 0.507                           0.503                          0.488                          0.490                 0.497
0.40    0.638            0.642                 0.687                           0.637                          0.661                          0.591                 0.627
0.50    0.744            0.736                 0.797                           0.725                  

## Verdict: the Phase 15 result survives hybrid, and survives the weight sweep

### The ranking does not change when BM25 joins

| model | semantic MRR | hybrid MRR (w=0.85) | Δ |
|---|---|---|---|
| **gemini-embedding-001** | 0.922 | **0.931** | +0.008 |
| openai-text-embedding-3-large | 0.837 | 0.838 | +0.001 |
| **cohere-embed-v4** *(incumbent)* | 0.828 | 0.838 | +0.010 |
| bge-m3 | 0.803 | 0.835 | +0.032 |
| multilingual-e5-large-instruct | 0.741 | 0.751 | +0.010 |
| openai-text-embedding-3-small | 0.694 | 0.692 | −0.002 |
| qwen3-embedding-0.6b | 0.657 | 0.674 | +0.017 |

Paired sign test on hybrid ranks — **gemini beats cohere 11–2, p = 0.0225**, slightly *stronger* than the semantic-only 12–3 / p = 0.035. Gemini also now separates from openai-3-large (12–3, p = 0.035). And openai-3-large vs cohere is **8–8, p = 1.000** — still a dead tie, confirming the semantic-only finding that the table ordering between those two means nothing.

### Gemini wins at every weight that uses the embeddings at all

| w | 0.0 | 0.2 | 0.4 | 0.6 | 0.8 | 0.85 | 1.0 |
|---|---|---|---|---|---|---|---|
| gemini | 0.437 | **0.507** | **0.687** | **0.864** | **0.931** | **0.931** | **0.922** |
| cohere | 0.437 | 0.506 | 0.642 | 0.761 | 0.844 | 0.838 | 0.828 |

This was the confound worth testing: `w = 0.85` was fitted in Phase 6 on clause chunks *with Cohere as the semantic ranker*, so using it to judge Gemini was not neutral. It turns out not to matter — **Gemini leads at every w from 0.2 to 1.0**, and the margin *widens* as the embedding gets more weight.

The decisive comparison is Gemini at Cohere's *own* best setting: at w = 0.80, where Cohere peaks at 0.844, Gemini scores **0.931**. Cohere's best configuration anywhere in the sweep (0.844) is still below Gemini's worst from w = 0.6 upward (0.864).

*(Reading note: at w = 0.0 all seven models are identical at 0.437 — that arm is pure BM25 and ignores embeddings entirely. The "winner at w=0.0" line reports bge-m3 only because `idxmax` breaks a seven-way tie alphabetically. It is a tie, not a win.)*

### BM25 is contributing very little here

**BM25 alone: recall@1 0.317, recall@5 0.600, MRR 0.437** — far below every embedding model. And fusing it in moves MRR by between −0.002 and +0.032, with Gemini gaining just +0.008.

Two consequences worth carrying forward:

1. **Hybrid is nearly free upside but not a rescue.** It never hurt the strong models meaningfully and helped the weak ones most (bge-m3 +0.032). The retrieval quality of this system is essentially the embedding model's quality.
2. **This weakens the earlier concern that natural phrasing would favour BM25.** It did not — v3's questions avoid statute vocabulary, which is exactly what BM25 needs, so the keyword half got *less* useful, not more.

### `w = 0.85` is no longer the best setting

Every model peaks at **w = 0.80 or 0.70**, none at 0.85. That confirms the RRF re-tune pending since Phase 12a is genuinely warranted — but the prize is small (Cohere 0.838 → 0.844; Gemini unchanged at 0.931), and it should be done *after* the embedding model is settled, not before, since the optimum moves with the model.

### Bottom line

Every check that could have overturned the Phase 15 result has now been run and none did:

- ✅ holds under production's exact hybrid configuration
- ✅ holds across the entire RRF weight range, including Cohere's own optimum
- ✅ statistically significant on paired tests in both semantic-only (p = 0.035) and hybrid (p = 0.0225)
- ✅ Gemini achieves it while *silently truncating* 4 articles that Cohere reads in full

The one metric that says otherwise — recall@5 tied at 1.000 — is saturated and cannot express a difference. **The evidence supports switching to `gemini-embedding-001`.**

Still not done unilaterally, and still not done in this notebook: switching means a new 3072-dim Pinecone index, an embedder abstraction in `src/retrieval.py`, and checking Gemini's production rate limits — the failure mode that removed Voyage in Phase 13. `src/retrieval.py` remains on Cohere `embed-v4.0`.
